In [2]:
pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.3 MB 3.3 MB/s eta 0:00:05
   --- ------------------------------------ 1.6/16.3 MB 4.0 MB/s eta 0:00:04
   ------ --------------------------------- 2.6/16.3 MB 4.5 MB/s eta 0:00:04
   --------- ------------------------------ 3.9/16.3 MB 5.1 MB/s eta 0:00:03
   ------------ --------------------------- 5.2/16.3 MB 5.2 MB/s eta 0:00:03
   ------------- -------------------------- 5.5/16.3 MB 4.9 MB/s eta 0:00:03
   -------------- ------------------------- 6.0/16.3 MB 4.5 MB/s eta 0:00:03
   ---------------- ----------------------- 6.6/16.3 MB 4.2 M


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_community.retrievers import WikipediaRetriever
from dotenv import load_dotenv

In [2]:
retriever = WikipediaRetriever(top_k_results=2, lang='en')

In [3]:
query = "History of World War 2 in perspective of Japan and America"
docs = retriever.invoke(query)

In [4]:
for i,doc in enumerate(docs):
    print(f"\n---------result {i+1}-----------")
    print(f"content : \n{doc.page_content}......")


---------result 1-----------
content : 
World War II, or the Second World War (1 September 1939 – 2 September 1945), was a global conflict between two coalitions: the Allies and the Axis powers. Nearly all of the world's countries participated, with many engaging in total war on an unprecedented scale. World War II was the deadliest conflict in history, causing the deaths of 60 to 75 million people, a majority of whom were civilians. Millions died as a result of massacres, starvation, disease, and genocides, including the Holocaust. After the Allied victory, Germany, Austria, Japan, and Korea were occupied, and German and Japanese leaders were tried for war crimes.
The causes of World War II included unresolved tensions in the aftermath of World War I and the rise of fascism in Europe and militarism in Japan. Key events preceding the war included Japan's invasion of Manchuria in 1931, the Spanish Civil War, the outbreak of the Second Sino-Japanese War in 1937, and Germany's annexation

## VectorStore as Retriever

In [5]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

In [11]:
documents = [
    Document(page_content="Mumbai is the Financial Capital of India and it belongs to Mahrashtra. It is known for Bollywood and is nearer to 'Pune City' one of the biggest IT hub of india."),
    Document(page_content="Bengaluru is the Silicon Valley of India where, most of the world's biggest tech players, as well as domestic tech giants, startups are located. It is  Capital of Karnataka Located in South India."),
    Document(page_content="Keralam is the tourism hub of the india located in south india and mainl known for Moutains, seashore."),
    Document(page_content="Hyderabad and Noida are also one of the tech cities or IT Hub in india located in Telangana and UttarPradesh located in South and North India Respectively."),
    Document(page_content="Delhi is a union territory and Capital of India. Punjab and Haryana which are neighbouring states to Delhi are one of the major agricultural hub of India")
]

In [12]:
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=OpenAIEmbeddings(),
    collection_name='My_Collection'
)

In [13]:
retriever1 = vector_store.as_retriever(search_kwargs = {"k":2})

In [14]:
query1 = "Which section in india is economically contributes more to its GDP?"
result1 = retriever1.invoke(query1)

In [15]:
for i, doc in enumerate(result1):
    print(f"\n------result{i+1}------")
    print(doc.page_content)


------result1------
Mumbai is the Financial Capital of India and it belongs to Mahrashtra. It is known for Bollywood and is nearer to 'Pune City' one of the biggest IT hub of india.

------result2------
Mumbai is the Financial Capital of India and it belongs to Mahrashtra. It is known for Bollywood and is nearer to 'Pune City' one of the biggest IT hub of india.


## Maximum Marginal Relevance Retriever

In [16]:
from langchain_community.vectorstores import FAISS

In [17]:
vector_store_faiss = FAISS.from_documents(
    documents=documents,
    embedding=OpenAIEmbeddings()
)

In [34]:
retriever_faiss = vector_store_faiss.as_retriever(search_type = "mmr", search_kwargs = {"k" : 3, "lambda_mult" : 0.3} )#lambda decides the level of MMR. lamda nearer to 0 -> More MMR, lamda nearer to 1 -> Less MMR. lambda is the tradeoff parameter
result2 = retriever_faiss.invoke(query1)

In [35]:
for i, doc in enumerate(result2):
        print(f"\n------result{i+1}------")
        print(doc.page_content)


------result1------
Mumbai is the Financial Capital of India and it belongs to Mahrashtra. It is known for Bollywood and is nearer to 'Pune City' one of the biggest IT hub of india.

------result2------
Delhi is a union territory and Capital of India. Punjab and Haryana which are neighbouring states to Delhi are one of the major agricultural hub of India

------result3------
Keralam is the tourism hub of the india located in south india and mainl known for Moutains, seashore.


## Multiuery Retriever

In [47]:
doc_mqr = [
     Document(
        page_content="Electric vehicles use electric motors instead of conventional petrol or diesel engines. They draw energy from rechargeable batteries."
    ),
    Document(
        page_content="Lithium-ion batteries are widely used in electric cars because they offer high energy density and can be recharged many times."
    ),
    Document(
        page_content="Regenerative braking allows an electric car to recover some energy while slowing down. The recovered energy is stored in the battery."
    ),
    Document(
        page_content="Public charging stations allow electric vehicle owners to recharge away from home. Fast chargers can replenish batteries much quicker than standard chargers."
    ),
    Document(
        page_content="A vehicle's driving range is the distance it can travel before its battery needs recharging. Cold weather, driving speed, and heating use can reduce this range."
    ),
    Document(
        page_content="Installing a home EV charger lets owners recharge their electric cars overnight. Charging speed depends on the charger's power output and the vehicle's capabilities."
    ),
    Document(
        page_content="Electric cars produce no tailpipe emissions while driving. However, their overall environmental impact also depends on battery manufacturing and how the electricity is generated."
    ),
    Document(
        page_content="Battery degradation gradually reduces an electric vehicle's energy storage capacity. As a result, an older EV may travel fewer kilometres on a full charge."
    )
]

In [48]:
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [49]:
vector_store1 = FAISS.from_documents(documents=doc_mqr, embedding=OpenAIEmbeddings())

In [57]:
similarity_retriever = vector_store1.as_retriever(search_type = "similarity", search_kwargs = {"k":3})

In [58]:
multiuery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store1.as_retriever(search_kwargs = {"k":3}),
    llm = ChatOpenAI(model="gpt-4.1-mini")
)

In [59]:
query2 = "Why does my electric car's driving range decrease over time, and what factors affect how far it can travel?"
similarity_results = similarity_retriever.invoke(query2)
multiuery_results = multiuery_retriever.invoke(query2)

In [60]:
for i, doc in enumerate(similarity_results):
    print(f"\n------result{i+1}------")
    print(doc.page_content)
    


------result1------
A vehicle's driving range is the distance it can travel before its battery needs recharging. Cold weather, driving speed, and heating use can reduce this range.

------result2------
Battery degradation gradually reduces an electric vehicle's energy storage capacity. As a result, an older EV may travel fewer kilometres on a full charge.

------result3------
Installing a home EV charger lets owners recharge their electric cars overnight. Charging speed depends on the charger's power output and the vehicle's capabilities.


In [61]:
for i, doc in enumerate(multiuery_results):
    print(f"\n------result{i+1}------")
    print(doc.page_content)


------result1------
A vehicle's driving range is the distance it can travel before its battery needs recharging. Cold weather, driving speed, and heating use can reduce this range.

------result2------
Battery degradation gradually reduces an electric vehicle's energy storage capacity. As a result, an older EV may travel fewer kilometres on a full charge.

------result3------
Electric cars produce no tailpipe emissions while driving. However, their overall environmental impact also depends on battery manufacturing and how the electricity is generated.

------result4------
Electric vehicles use electric motors instead of conventional petrol or diesel engines. They draw energy from rechargeable batteries.

------result5------
Public charging stations allow electric vehicle owners to recharge away from home. Fast chargers can replenish batteries much quicker than standard chargers.


## Contextual Compression Retriever

In [62]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [73]:
from langchain_core.documents import Document

docs_con = [
    Document(
        page_content=(
            "Drinking coffee late in the afternoon can interfere with nighttime sleep. "
            "Caffeine blocks adenosine receptors in the brain, reducing sleepiness. "
            "Caffeine can remain in the body for several hours, depending on individual metabolism. "
            "Coffee beans are harvested in tropical regions and are often roasted at high temperatures. "
            "Dark roasts generally have a stronger roasted flavor than light roasts."
        ),  metadata={"source": "Doc1"}
    ),
    Document(
        page_content=(
            "Caffeine is a natural stimulant found in coffee, tea, and energy drinks. "
            "It promotes alertness by blocking adenosine receptors associated with sleep pressure. "
            "Consuming caffeine close to bedtime can delay sleep and reduce sleep quality. "
            "Espresso is prepared by forcing hot water through finely ground coffee beans. "
            "Many cafes serve espresso with milk to prepare cappuccinos and lattes."
        ),  metadata={"source": "Doc2"}
    ),
    Document(
        page_content=(
            "Regular exercise can improve sleep quality and reduce stress. "
            "Walking, cycling, and swimming are common aerobic activities. "
            "Physical activity supports cardiovascular health and muscle endurance. "
            "Coffee contains various aromatic compounds that develop during roasting. "
            "Arabica and Robusta are two widely cultivated coffee species."
        ),  metadata={"source": "Doc3"}
    )
]

In [74]:
vector_store2 = FAISS.from_documents(documents=docs_con, embedding=OpenAIEmbeddings())

In [75]:
base_retriever = vector_store2.as_retriever(search_kwargs = {"k":2})

In [76]:
compressor = LLMChainExtractor.from_llm(llm=ChatOpenAI(model ='gpt-4.1-mini'))

In [77]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [80]:
query3 = "How can I improve concentration when working?"
compressed_res = compression_retriever.invoke(query3)

In [81]:
for i, doc in enumerate(compressed_res):
    print(f"\n------result{i+1}------")
    print(doc.page_content)
    


------result1------
Regular exercise can improve sleep quality and reduce stress.

------result2------
Caffeine is a natural stimulant found in coffee, tea, and energy drinks. It promotes alertness by blocking adenosine receptors associated with sleep pressure.
